# Install and Run Perseus MCP



## Table of contents

* <a href="#introduction">1 - Introduction</a>
* <a href="#understand">2 - Understand the installation choices</a>
* <a href="#install_pypi">3 - Install from PyPI with pip</a>
* <a href="#install_uv">4 - Install and run with uv</a>
* <a href="#clone">5 - Clone and run the repository locally</a>
* <a href="#choose_launch">6 - Choose a launch command</a>
* <a href="#configure_client">7 - Configure an MCP client</a>
* <a href="#verify">8 - Verify the installed package</a>
* <a href="#mcp_inspector">9 - Test with MCP Inspector</a>
* <a href="#update">10 - Update or uninstall</a>
* <a href="#troubleshoot">11 - Troubleshoot common setup problems</a>
* <a href="#continue">12 - Continue with the research notebooks</a>
* <a href="#required-libraries">13 - Required libraries</a>
* <a href="#mcp-server">14 - MCP server</a>
* <a href="#notebook-version">15 - Notebook version</a>

## 1 - Introduction <a class="anchor" id="introduction"></a>
##### [Back to ToC](#TOC)

This notebook explains the supported ways to install, launch, configure, update, and remove the Perseus MCP server. It is an onboarding guide rather than a text-research workflow; continue with notebook `01_` or `03_` after choosing an installation method.

## 2 - Understand the installation choices <a class="anchor" id="understand"></a>
##### [Back to ToC](#TOC)

Perseus MCP is a **local stdio MCP server**. An MCP-capable application launches the server as a child process and communicates with it over standard input and output. The server contacts the public Perseus CTS and Scaife services; it does not download the full corpus and does not require an API key.

| Method | Best for | What the client launches |
|---|---|---|
| PyPI in a virtual environment | Most users and stable releases | `perseus-mcp` or `python -m perseus_mcp` |
| `uv tool install` | An isolated command available on your PATH | `perseus-mcp` |
| `uvx` | Trying the latest published package without a persistent install | `uvx perseus-mcp` |
| Editable repository install | Contributors changing code or notebooks | The editable `perseus-mcp` command |
| Repository-local `uv run` | Running directly from a clone | `uv --directory ... run perseus-mcp` |

Use only one method at first. Multiple installations can make it unclear which executable an MCP client is launching.

## 3 - Install from PyPI with pip <a class="anchor" id="install_pypi"></a>
##### [Back to ToC](#TOC)

A virtual environment keeps Perseus MCP and its dependencies separate from other Python projects.

### macOS or Linux

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install perseus-mcp
```

### Windows PowerShell

```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install perseus-mcp
```

The installation provides both a `perseus-mcp` console command and the Python module entry point `python -m perseus_mcp`.

## 4 - Install and run with uv <a class="anchor" id="install_uv"></a>
##### [Back to ToC](#TOC)

[uv](https://docs.astral.sh/uv/) can install the command into an isolated tool environment:

```bash
uv tool install perseus-mcp
perseus-mcp
```

To try the published package without keeping a tool installation:

```bash
uvx perseus-mcp
```

`uvx` is convenient for testing, but an explicit installed command or absolute interpreter path is more predictable in long-lived desktop-client configuration.

## 5 - Clone and run the repository locally <a class="anchor" id="clone"></a>
##### [Back to ToC](#TOC)

Use this method when developing the server, running the repository notebooks, or testing unreleased changes.

```bash
git clone https://github.com/tonyjurg/Perseus-mcp.git
cd Perseus-mcp
uv sync
uv run perseus-mcp
```

An editable pip installation is equivalent for development:

```bash
python -m venv .venv
# Activate the environment, then:
python -m pip install -e ".[dev]"
perseus-mcp
```

The implementation lives in `src/perseus_mcp/server.py`. Launch it through the installed `perseus-mcp` command or the `python -m perseus_mcp` module entry point.

## 6 - Choose a launch command <a class="anchor" id="choose_launch"></a>
##### [Back to ToC](#TOC)

Running the server directly normally appears to wait without printing a prompt. That is expected: stdio MCP servers wait for an MCP client to send protocol messages. Stop a manual run with `Ctrl+C`.

Supported launch forms:

```bash
# Installed console command
perseus-mcp

# Installed package through a particular interpreter
python -m perseus_mcp

# Repository-local uv environment
uv --directory /full/path/to/Perseus-mcp run perseus-mcp

# Published package in an ephemeral uv environment
uvx perseus-mcp
```

For desktop applications, `python -m perseus_mcp` with an **absolute path to the virtual environment's Python executable** is usually the least ambiguous option.

## 7 - Configure an MCP client <a class="anchor" id="configure_client"></a>
##### [Back to ToC](#TOC)

Most MCP clients expect a server name, executable, argument list, and optional environment variables.

### Installed package using an absolute Python path

```json
{
  "mcpServers": {
    "perseus": {
      "command": "/absolute/path/to/.venv/bin/python",
      "args": ["-m", "perseus_mcp"],
      "env": {}
    }
  }
}
```

On Windows, `command` will resemble `C:\\full\\path\\to\\.venv\\Scripts\\python.exe`.

### Repository clone using uv

```json
{
  "mcpServers": {
    "perseus": {
      "command": "uv",
      "args": [
        "--directory",
        "/full/path/to/Perseus-mcp",
        "run",
        "perseus-mcp"
      ],
      "env": {}
    }
  }
}
```

Replace all example paths with absolute paths. Restart the MCP client after changing its configuration.

Run the next cell from a repository checkout to install Perseus MCP into the Python environment used by this notebook. Set `PERSEUS_MCP_INSTALL_SOURCE` in that cell to `"repo"` for editable development-branch work or `"pypi"` for the published package. The generated client configuration below then uses this same Python executable.

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

# Use "repo" while working on this development checkout.
# Use "pypi" to run against the published package normal users install.
PERSEUS_MCP_INSTALL_SOURCE = "repo"  # "repo" or "pypi"
PERSEUS_MCP_PYPI_SPEC = "perseus-mcp"

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "pyproject.toml").exists()
        and (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

install_source = PERSEUS_MCP_INSTALL_SOURCE.lower()
if install_source == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find the Perseus-mcp repository from {START}. Open this notebook inside the repository checkout or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    package_target = ["--editable", str(REPO_ROOT)]
    package_label = f"editable repository at {REPO_ROOT}"
elif install_source == "pypi":
    package_target = ["--force-reinstall", PERSEUS_MCP_PYPI_SPEC]
    package_label = PERSEUS_MCP_PYPI_SPEC
else:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", *package_target]
)

if REPO_ROOT is not None:
    repo_src_dir = (REPO_ROOT / "src").resolve()

    def _is_repo_src(path_entry):
        try:
            return Path(path_entry).resolve() == repo_src_dir
        except (OSError, RuntimeError):
            return False

    sys.path = [path_entry for path_entry in sys.path if not _is_repo_src(path_entry)]

if install_source == "repo":
    src_dir = str(REPO_ROOT / "src")
    if src_dir not in sys.path:
        sys.path.insert(0, src_dir)

importlib.invalidate_caches()

print(f"Installed perseus-mcp from {package_label} into this kernel")

In [ ]:
import json
import sys

mcp_config = {
    "mcpServers": {
        "perseus": {
            "command": sys.executable,
            "args": ["-m", "perseus_mcp"],
            "env": {},
        }
    }
}

print(json.dumps(mcp_config, indent=2))

## 8 - Verify the installed package <a class="anchor" id="verify"></a>
##### [Back to ToC](#TOC)

After running the current-kernel installation cell above, run the next cell in the same notebook kernel. In `"repo"` mode it verifies that the editable checkout is the import source. In `"pypi"` mode it verifies that the installed package is not being shadowed by the local `src` directory. It does not start the server or make network requests.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path
from shutil import which

install_source = globals().get("PERSEUS_MCP_INSTALL_SOURCE", "repo").lower()
if install_source not in {"repo", "pypi"}:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

try:
    installed_version = version("perseus-mcp")
except PackageNotFoundError:
    installed_version = None

package_spec = find_spec("perseus_mcp")
console_command = which("perseus-mcp")
package_origin = Path(package_spec.origin).resolve() if package_spec and package_spec.origin else None

print(f"Install source selected: {install_source}")
print(f"Installed distribution version: {installed_version or 'not installed'}")
print(f"Importable package location: {package_origin or 'not importable'}")
print(f"Console command: {console_command or 'not found on PATH'}")

assert installed_version is not None, (
    "Run the current-kernel installation cell first. An editable repo install also creates distribution metadata."
)
assert package_origin is not None, "perseus_mcp is installed but not importable in this running kernel."

repo_root = globals().get("REPO_ROOT")
if repo_root is None:
    start = Path.cwd().resolve()
    repo_root = next(
        (
            candidate
            for candidate in [start, *start.parents]
            if (candidate / "src" / "perseus_mcp" / "server.py").exists()
        ),
        None,
    )
else:
    repo_root = Path(repo_root)

if install_source == "repo":
    assert repo_root is not None, "Repo source selected, but the repository root was not found."
    repo_src = (repo_root / "src").resolve()
    assert package_origin.is_relative_to(repo_src), (
        f"Expected repo source under {repo_src}, but imported {package_origin}."
    )
elif repo_root is not None:
    repo_src = (repo_root / "src").resolve()
    assert not package_origin.is_relative_to(repo_src), (
        f"PyPI source selected, but local repo source is shadowing the install: {package_origin}."
    )

In [ ]:
import importlib.metadata
import sys

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

try:
    version = importlib.metadata.version("perseus-mcp")
except importlib.metadata.PackageNotFoundError:
    print("perseus-mcp distribution: not installed in this kernel")
else:
    print(f"perseus-mcp distribution version: {version}")

try:
    import perseus_mcp
    from perseus_mcp import server as perseus_server
except ModuleNotFoundError as exc:
    print(f"perseus_mcp import: failed ({exc})")
else:
    print(f"perseus_mcp package: {perseus_mcp.__file__}")
    print(f"perseus_mcp.server import: OK ({perseus_server.__file__})")

## 9 - Test with MCP Inspector <a class="anchor" id="mcp_inspector"></a>
##### [Back to ToC](#TOC)

MCP Inspector starts the server, performs the MCP handshake, and provides a browser interface for listing and calling tools.

For an installed command:

```bash
npx @modelcontextprotocol/inspector perseus-mcp
```

For a repository clone:

```bash
npx @modelcontextprotocol/inspector uv --directory /full/path/to/Perseus-mcp run perseus-mcp
```

After connecting, confirm that tools such as `find_author_names`, `get_author_resources`, `get_passage_plaintext`, and `search_perseus` are listed.

## 10 - Update or uninstall <a class="anchor" id="update"></a>
##### [Back to ToC](#TOC)

### pip installation

```bash
python -m pip install --upgrade perseus-mcp
python -m pip uninstall perseus-mcp
```

### uv tool installation

```bash
uv tool upgrade perseus-mcp
uv tool uninstall perseus-mcp
```

### Repository clone

```bash
git pull
uv sync
```

Restart the MCP client after upgrading so it launches the new process.

## 11 - Troubleshoot common setup problems <a class="anchor" id="troubleshoot"></a>
##### [Back to ToC](#TOC)

| Symptom | Likely cause and response |
|---|---|
| `perseus-mcp` is not found | The environment is not activated or its scripts directory is not on `PATH`; use the absolute Python path with `-m perseus_mcp` |
| `No module named perseus_mcp` | The client is launching a different Python interpreter; install into that interpreter or correct `command` |
| The command appears to hang | Normal for a stdio server waiting for MCP messages; test with MCP Inspector instead |
| The client shows no tools | Restart it, validate JSON syntax, use absolute paths, and run the same command manually to expose startup errors |
| A local code edit is ignored | The client is launching the PyPI installation instead of the editable checkout, or a stale process is still running |
| Cache files appear in an unexpected directory | Set `PERSEUS_MCP_CACHE_DIR` to an absolute writable location in the client `env` object |
| `429 Too Many Requests` from `perseus.tufts.edu` | Too many CTS calls were sent in a short period. Stop the batch, wait before retrying, reduce concurrency, and add a delay between passage calls. The server currently does not retry 429 responses automatically |
| Passage or search calls fail after connection | The local server is running, but the public Perseus or Scaife service may be unavailable or the URN/query may be invalid |

The Perseus MCP server itself needs no OpenRouter, Anthropic, OpenAI, or other model-provider key. Model credentials belong to the MCP host or to optional LLM demonstration notebooks.

## 12 - Continue with the research notebooks <a class="anchor" id="continue"></a>
##### [Back to ToC](#TOC)

- [`01_basic_cts_workflow.ipynb`](01_basic_cts_workflow.ipynb) explains the underlying CTS HTTP service.
- [`02_search_and_navigation.ipynb`](02_search_and_navigation.ipynb) explains direct Perseus and Scaife search concepts.
- [`03_mcp_connection_homer_iliad.ipynb`](03_mcp_connection_homer_iliad.ipynb) makes the first in-process MCP tool calls.
- [`05_mcp_all_tools.ipynb`](05_mcp_all_tools.ipynb) catalogs the complete live MCP tool surface.

Project resources:

- [PyPI package](https://pypi.org/project/perseus-mcp/)
- [GitHub repository](https://github.com/tonyjurg/Perseus-mcp)
- [End-user guide](https://tonyjurg.github.io/Perseus-mcp/enduser/)
- [MCP Inspector](https://github.com/modelcontextprotocol/inspector)

## 13 - Required libraries <a class="anchor" id="#required-libraries"></a>
##### [Back to ToC](#TOC)

This repository targets **Python 3.11 or newer**. Installing the local repository with the current-kernel cell above installs the `perseus-mcp` package and its declared runtime dependencies into the active Jupyter environment. The package dependencies include `fastmcp>=2.12.0`, `httpx>=0.27.0`, and `defusedxml>=0.7.1`.

The `json`, `sys`, `pathlib`, `subprocess`, `importlib.metadata`, and `tomllib` modules used by this installation notebook are included with Python. Jupyter/IPython is also needed to run the cells.

The equivalent command from the repository root is:

```bash
pip install -e .
```

## 14 - MCP server version <a class="anchor" id="mcp-version"></a>
##### [Back to ToC](#TOC)

In [ ]:
import importlib.metadata
import sys
import tomllib
from pathlib import Path

try:
    perseus_mcp_version = importlib.metadata.version("perseus-mcp")
except importlib.metadata.PackageNotFoundError:
    pyproject_path = next(
        (
            parent / "pyproject.toml"
            for parent in [Path.cwd(), *Path.cwd().parents]
            if (parent / "pyproject.toml").exists()
        ),
        None,
    )

    if pyproject_path is None:
        perseus_mcp_version = "not installed; pyproject.toml not found"
    else:
        with pyproject_path.open("rb") as f:
            perseus_mcp_version = tomllib.load(f)["project"]["version"]

print(f"perseus-mcp version: {perseus_mcp_version}")
print(f"Python version: {sys.version.split()[0]}")

## 15 - Notebook version <a class="anchor" id="notebook-version"></a>
##### [Back to ToC](#TOC)

| Field | Value |
|---|---|
| Author | Tony Jurg |
| Notebook Version | 1.2 |
| Date | June 26, 2026 |